# Carga de imagenes al datastore y mosaic dataset

Flujo operativo posterior a la preparacion del notebook `008`. Usa `04_ready_for_datastore.csv` para copiar archivos al datastore y `06_attribute_updates.csv` para actualizar atributos del mosaic dataset.

In [1]:
from datetime import datetime
from pathlib import Path
import importlib

import pandas as pd

import core.mosaic_loader as mosaic_loader
mosaic_loader = importlib.reload(mosaic_loader)
from core.mosaic_loader import *

# PARAMETROS
RUN_PREPARACION = "20260615_155625"
OUTPUT_PREPARACION_DIR = Path.cwd() / "outputs" / "preparacion_carga_mosaico" / RUN_PREPARACION

READY_FOR_DATASTORE_CSV = OUTPUT_PREPARACION_DIR / "04_ready_for_datastore.csv"
ATTRIBUTE_UPDATES_CSV = OUTPUT_PREPARACION_DIR / "06_attribute_updates.csv"

PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"

# Seguridad operacional: validar primero con DRY_RUN=True. Cambiar a False para ejecutar copia/carga/update.
DRY_RUN = False
OVERWRITE_COPY = False
SKIP_EXISTING_MOSAIC_NAME = True

# Valores fijos definidos para esta carga.
MAXPS_VALUE = 10000
LOWPS_VALUE = 0.15

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "outputs" / "carga_mosaico" / run_timestamp
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Preparacion:", OUTPUT_PREPARACION_DIR)
print("CSV carga:", READY_FOR_DATASTORE_CSV)
print("CSV atributos:", ATTRIBUTE_UPDATES_CSV)
print("Mosaic dataset:", PATH_MOSAIC_DATASET)
print("Salida:", OUTPUT_DIR)
print("DRY_RUN:", DRY_RUN)

Preparacion: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_155625
CSV carga: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_155625\04_ready_for_datastore.csv
CSV atributos: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\preparacion_carga_mosaico\20260615_155625\06_attribute_updates.csv
Mosaic dataset: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport
Salida: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\carga_mosaico\20260615_162625
DRY_RUN: False


## 1. Cargar manifiestos

Se valida que cada imagen lista tenga atributos asociados antes de ejecutar cualquier operacion.

In [2]:
load_df = load_ready_and_attributes(READY_FOR_DATASTORE_CSV, ATTRIBUTE_UPDATES_CSV)

required_columns = ["path", "destination_path", "Name", "Sector", "Fecha_Adqui", "URL", "Proyecto", "Sensor", "Fecha_Publ"]
missing_columns = [column for column in required_columns if column not in load_df.columns]
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_columns}")

validation_summary = pd.DataFrame(
    [
        {"metric": "records_to_process", "value": len(load_df)},
        {"metric": "missing_source_path", "value": int((~load_df["path"].map(lambda value: Path(value).exists())).sum())},
        {"metric": "missing_destination_path", "value": int(load_df["destination_path"].isna().sum())},
        {"metric": "unique_destination_paths", "value": int(load_df["destination_path"].nunique())},
        {"metric": "unique_names", "value": int(load_df["Name"].nunique())},
    ]
)

display(validation_summary)
display(load_df[["file_name", "Name", "destination_path", "Sector", "Fecha_Adqui", "Proyecto", "Sensor", "Fecha_Publ"]].head(20))

,metric,value
0,records_to_process,32
1,missing_source_path,0
2,missing_destination_path,0
3,unique_destination_paths,32
4,unique_names,32


,file_name,Name,destination_path,Sector,Fecha_Adqui,Proyecto,Sensor,Fecha_Publ
0,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_D...,Estacion_Cabecera,2026-05-01,PAO,DJI Mavic Enterprise,2026-06-15
1,GEOSP-TRN-002545_GS_ORTOFOTO_EB3_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,Subestacion-El-Mauro,2026-05-06,PAO,DJI Mavic Enterprise,2026-06-15
2,GEOSP-TRN-002546_GS_ORTOFOTO_SSEE_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,Subestacion-El-Mauro,2026-05-06,PAO,DJI Mavic Enterprise,2026-06-15
3,GEOSP-TRN-002555_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,TORRE_E85_A_E_125,2026-05-06,PAO,DJI Mavic Enterprise,2026-06-15
4,GEOSP-TRN-002591_GS_Ortofoto ED1_10-05-2026.tif,CL_MLP_PAO_IF_Ortho_26_05_10_ED1,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ED1,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
5,GEOSP-TRN-002592_GS_Ortofoto_Helipuerto Mauro ...,CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,Helipuerto,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
6,GEOSP-TRN-002593_GS_ORTOFOTO_PATIO 19B_10-05-2...,CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,Patio-19B-y-Armado,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
7,GEOSP-TRN-002603_ORTOFOTO_CORTADA_EM2_100526.tif,CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,DME9-PA12-IIFF8,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15
8,GEOSP-TRN-002604_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,TORRES_E48_A_E84_PV4,2026-05-07,PAO,DJI Mavic Enterprise,2026-06-15
9,GEOSP-TRN-002606_GS_Ortofoto_Tramo 1 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2026-05-10,PAO,DJI Mavic Enterprise,2026-06-15


## 2. Ejecutar copia, carga al mosaico, footprints y atributos

Con `DRY_RUN=True` no escribe archivos ni modifica el mosaic dataset. Con `DRY_RUN=False` ejecuta el flujo completo por imagen.

In [3]:
results = []

for index, row in load_df.iterrows():
    print(f"[{index + 1}/{len(load_df)}] {row['Name']}")
    result = process_mosaic_load_row(
        row,
        PATH_MOSAIC_DATASET,
        overwrite_copy=OVERWRITE_COPY,
        skip_existing_mosaic_name=SKIP_EXISTING_MOSAIC_NAME,
        maxps_value=MAXPS_VALUE,
        lowps_value=LOWPS_VALUE,
        dry_run=DRY_RUN,
    )
    results.append(result)

results_df = pd.DataFrame(results)
display(results_df)
display(results_df["overall_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "overall_status"}))

[1/32] CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera
[2/32] CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-1
[3/32] CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-2
[4/32] CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125
[5/32] CL_MLP_PAO_IF_Ortho_26_05_10_ED1
[6/32] CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto
[7/32] CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado
[8/32] CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8
[9/32] CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4
[10/32] CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-Km-84p2-a-82p3
[11/32] CL_MLP_PAO_IF_Ortho_26_05_13_DME9-PA12-IIFF8
[12/32] CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro_A_E35
[13/32] CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-1
[14/32] CL_MLP_PAO_IF_Ortho_26_05_13_EM2_S2
[15/32] CL_MLP_PAO_IF_Ortho_26_05_13_EBD-1
[16/32] CL_MLP_PAO_IF_Ortho_26_05_13_ED2
[17/32] CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-2
[18/32] CL_MLP_PAO_IF_Ortho_26_05_13_EBD-2
[19/32] CL_MLP_PAO_IF_Ortho_26_05_13_Estacion_

,file_name,Name,destination_path,overall_status,source_path,copy_status,copy_error,mosaic_add_status,mosaic_add_error,footprint_status,footprint_error,attribute_status,attribute_rows_updated,attribute_missing_fields,attribute_error
0,GEOSP-TRN-002511_GS_Ortofoto Estación Cabecera...,CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_D...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
1,GEOSP-TRN-002545_GS_ORTOFOTO_EB3_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
2,GEOSP-TRN-002546_GS_ORTOFOTO_SSEE_06-05-26.tif,CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Ma...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
3,GEOSP-TRN-002555_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
4,GEOSP-TRN-002591_GS_Ortofoto ED1_10-05-2026.tif,CL_MLP_PAO_IF_Ortho_26_05_10_ED1,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
5,GEOSP-TRN-002592_GS_Ortofoto_Helipuerto Mauro ...,CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\El_Mauro...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
6,GEOSP-TRN-002593_GS_ORTOFOTO_PATIO 19B_10-05-2...,CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
7,GEOSP-TRN-002603_ORTOFOTO_CORTADA_EM2_100526.tif,CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
8,GEOSP-TRN-002604_GS_Ortofoto_Tramo 2 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None
9,GEOSP-TRN-002606_GS_Ortofoto_Tramo 1 Línea 33 ...,CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,ok,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,copied,None,added,None,built,None,updated,1,Fecha_Adqui|Fecha_Publ,None


,overall_status,count
0,ok,32


## 3. Exportar resultados de ejecucion

In [4]:
summary_rows = [
    {"metric": "run_timestamp", "value": run_timestamp},
    {"metric": "preparation_run", "value": RUN_PREPARACION},
    {"metric": "dry_run", "value": DRY_RUN},
    {"metric": "mosaic_dataset", "value": PATH_MOSAIC_DATASET},
    {"metric": "records_to_process", "value": len(load_df)},
    {"metric": "maxps_value", "value": MAXPS_VALUE},
    {"metric": "lowps_value", "value": LOWPS_VALUE},
]

for column in ["copy_status", "mosaic_add_status", "footprint_status", "attribute_status", "overall_status"]:
    if column in results_df.columns:
        for status, count in results_df[column].value_counts(dropna=False).items():
            summary_rows.append({"metric": f"{column}_{status}", "value": int(count)})

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_DIR / "00_summary.csv"
results_csv = OUTPUT_DIR / "01_load_results.csv"
errors_csv = OUTPUT_DIR / "02_errors.csv"

summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
results_df.to_csv(results_csv, index=False, encoding="utf-8-sig")
error_columns = [column for column in results_df.columns if column.endswith("_error")]
error_filter = results_df[error_columns].notna().any(axis=1) if error_columns else pd.Series(False, index=results_df.index)
results_df[error_filter].to_csv(errors_csv, index=False, encoding="utf-8-sig")

display(summary_df)
print("Resultados exportados en:", OUTPUT_DIR)

,metric,value
0,run_timestamp,20260615_162625
1,preparation_run,20260615_155625
2,dry_run,False
3,mosaic_dataset,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
4,records_to_process,32
5,maxps_value,10000
6,lowps_value,0.15
7,copy_status_copied,32
8,mosaic_add_status_added,32
9,footprint_status_built,32


Resultados exportados en: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\carga_mosaico\20260615_162625


In [5]:
Name = 'CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera'
load_df['Name'].tolist()

['CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera',
 'CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-1',
 'CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-2',
 'CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125',
 'CL_MLP_PAO_IF_Ortho_26_05_10_ED1',
 'CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto',
 'CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado',
 'CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8',
 'CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4',
 'CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-Km-84p2-a-82p3',
 'CL_MLP_PAO_IF_Ortho_26_05_13_DME9-PA12-IIFF8',
 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro_A_E35',
 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-1',
 'CL_MLP_PAO_IF_Ortho_26_05_13_EM2_S2',
 'CL_MLP_PAO_IF_Ortho_26_05_13_EBD-1',
 'CL_MLP_PAO_IF_Ortho_26_05_13_ED2',
 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-2',
 'CL_MLP_PAO_IF_Ortho_26_05_13_EBD-2',
 'CL_MLP_PAO_IF_Ortho_26_05_13_Estacion_Intermedia',
 'CL_MLP_PAO_IF_Ortho_26_05_14_EM3',
 'CL_MLP_PAO_IF_Ort

In [7]:
q = tuple(results_df['Name'])
q = f'Name IN {q}'
q

"Name IN ('CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera', 'CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-1', 'CL_MLP_PAO_IF_Ortho_26_05_06_Subestacion-El-Mauro-2', 'CL_MLP_PAO_IF_Ortho_26_05_06_TORRE_E85_A_E_125', 'CL_MLP_PAO_IF_Ortho_26_05_10_ED1', 'CL_MLP_PAO_IF_Ortho_26_05_10_Helipuerto', 'CL_MLP_PAO_IF_Ortho_26_05_10_Patio-19B-y-Armado', 'CL_MLP_PAO_IF_Ortho_26_05_10_DME9-PA12-IIFF8', 'CL_MLP_PAO_IF_Ortho_26_05_07_TORRES_E48_A_E84_PV4', 'CL_MLP_PAO_IF_Ortho_26_05_10_MonteAranda-NSTC-Km-84p2-a-82p3', 'CL_MLP_PAO_IF_Ortho_26_05_13_DME9-PA12-IIFF8', 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro_A_E35', 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-1', 'CL_MLP_PAO_IF_Ortho_26_05_13_EM2_S2', 'CL_MLP_PAO_IF_Ortho_26_05_13_EBD-1', 'CL_MLP_PAO_IF_Ortho_26_05_13_ED2', 'CL_MLP_PAO_IF_Ortho_26_05_13_Subestacion-El-Mauro-2', 'CL_MLP_PAO_IF_Ortho_26_05_13_EBD-2', 'CL_MLP_PAO_IF_Ortho_26_05_13_Estacion_Intermedia', 'CL_MLP_PAO_IF_Ortho_26_05_14_EM3', 'CL_MLP_PAO_IF_Ortho_26_05_14

In [ ]:
import arcpy

Estado = 'Activo'
with arcpy.da.SearchCursor(
    PATH_MOSAIC_DATASET,
    ['NombreVuelo'],
    where_clause=None,
    sql_clause=(None, "ORDER BY Name ASC")
) as cursor:
    for c in cursor:
        if c[0] is not None:
            print(c)
            break

('143_25_12_19_Plataforma_Integrada_El_Mauro',)


In [ ]:
# with arcpy.da.UpdateCursor(
#     PATH_MOSAIC_DATASET,
#     ['Estado'],
#     where_clause=q
    
# ) as cursor:
#     for c in cursor:
#         c[0] = 'Activo'
#         cursor.updateRow(c)

['Activo']

## Se cargan los footprint

In [54]:
arcpy.env.overwriteOutput = True
fc_footprint_temp = 'in_memory/foot3'
arcpy.ExportMosaicDatasetGeometry_management(PATH_MOSAIC_DATASET,
                                             out_feature_class=fc_footprint_temp,
                                             where_clause=q)


<Result 'in_memory\\foot3'>

In [55]:
arcpy.GetCount_management(fc_footprint_temp)[0]

'32'

In [56]:
import re
fc_footprint = r'\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO'

mapsfields = {
    'FechaAdqui':'Fecha_Adqu', 
    'FechaCarga':'Fecha_Publ' ,
    'NombreVuelo':'Nombre_de_Vuelo', 
    'ProductName':'ProductNam'
    }
## Se renombran los campos de la exportacion
for f in mapsfields:
    old_name = f
    new_name = mapsfields[f]
    print(f'{old_name} ==> {new_name}')
    arcpy.AlterField_management(fc_footprint_temp,old_name,new_name)

FechaAdqui ==> Fecha_Adqu
FechaCarga ==> Fecha_Publ
NombreVuelo ==> Nombre_de_Vuelo
ProductName ==> ProductNam


In [57]:
arcpy.Append_management(fc_footprint_temp,fc_footprint,'NO_TEST','')

<Result '\\\\amssclgis08.ams.gmams.cl\\CL_MLP_PAO\\02_FGDB\\CL_MLP_PAO_v1.gdb\\CL_MLP_PAO_06_COMPLEMENTOS\\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO'>

In [58]:
len([c for c in arcpy.da.SearchCursor(fc_footprint,['Nombre_de_Vuelo'],"Nombre_de_Vuelo is null")])

36

In [73]:
cols = ['Nombre_de_Vuelo','Sector','Fecha_Adqu']
with arcpy.da.UpdateCursor(
    fc_footprint,
    cols,
    where_clause=None,
    sql_clause=(None, "ORDER BY Name ASC")
) as cursor:
    cnt = 0
    for c in cursor:
        if c[1] and c[2]:
            cnt +=1
            fecha = pd.to_datetime(c[2]).strftime('%y_%m_%d')
            numero = str(cnt).zfill(3)
            sector = c[1]
            name_ = f'{numero}_{fecha}_{sector}'
            c[0] = name_
            cursor.updateRow(c)
            print(name_)
       

001_25_01_08_EB1
002_25_02_26_RutaD835
003_25_03_10_NSTC_116a118
004_25_03_10_NSTC_118a120
005_25_03_12_SRA2
006_25_03_17_NSTC_Km_120_a_121
007_25_03_19_Orejas16_Ruta-D-865
008_25_03_19_Orejas17_Ruta-D-865
009_25_03_19_Orejas18
010_25_03_31_EB2
011_25_03_31_EV2
012_25_04_02_DME-13
013_25_04_02_Patio-Acopio-17
014_25_04_02_Ruta-SE-a-DME-13
015_25_04_03_Area-patio-19b-y-armado
016_25_04_03_Subestacion-El-Mauro
017_25_04_09_Cachimba_de_Bajo_Camisas_ED2
018_25_04_10_Sector_Pupio_I_Area_1
019_25_05_07_Monte-Aranda-84p2-a-82p3
020_25_05_07_Monte_Aranda_82p3_a_80p7
021_25_05_12_EDT
022_25_05_12_IIFF15-Campamento-Tipay
023_25_05_15_DME9_PA12_IIFF8
024_25_05_15_ED1_IIFF7_DME8
025_25_05_15_EM2_PA11_01
026_25_05_29_DME5A_DME17_a_NSTC_78p9
027_25_06_02_ByPassDrenes_Estacion_Drenes_El_Mauro
028_25_06_02_Estacion_Drenes_El_Mauro
029_25_06_04_CaminoAcceso03_LasAnimas
030_25_06_04_EM3
031_25_06_05_33-kv-74p7-a-75p5
032_25_06_05_33-kV-NSTC-km-75p5-a-76p4
033_25_06_09_EB2
034_25_06_09_EV2
035_25_06_09_E

In [80]:
## Se actualiza en los footprint
querymosaic = [c[0] for c in arcpy.da.SearchCursor(PATH_MOSAIC_DATASET,['Name'])]

In [81]:
# querymosaic = f"Name IN {tuple(querymosaic)}"
cursor = arcpy.da.SearchCursor(fc_footprint,['Name','Nombre_de_Vuelo'])
df = pd.DataFrame(cursor,columns=['Name','Nombre_de_Vuelo'])
print(f'total de registros {df.shape[0]}')
df = df[df.Name.isin(querymosaic)]
print(f'total de registros {df.shape[0]}') 
df.head()

total de registros 483
total de registros 340


,Name,Nombre_de_Vuelo
141,CL_MLP_PAO_IF_Ortho_26_01_03_DME5A-DME17-a-NST...,145_26_01_03_DME5A-DME17-a-NSTC-78p9
142,CL_MLP_PAO_IF_Ortho_26_01_10_MonteAranda-NSTC-...,156_26_01_10_MonteAranda-NSTC-Km-84p2-a-82p3
143,CL_MLP_PAO_IF_Ortho_26_01_17_MonteAranda-NSTC-...,165_26_01_17_MonteAranda-NSTC-Km-84p2-a-82p3
144,CL_MLP_PAO_IF_Ortho_26_01_21_InstalacionesTipay,171_26_01_21_InstalacionesTipay
145,CL_MLP_PAO_IF_Ortho_26_01_11_EM1,157_26_01_11_EM1


In [98]:
result = []
with arcpy.da.SearchCursor(PATH_MOSAIC_DATASET,['Name','NombreVuelo']) as cursor:
    for c in cursor:
        try:
            name = c[0]
            nombre_vuelo = df.loc[df.Name == name,'Nombre_de_Vuelo'].values[0]
            result.append([c[1],nombre_vuelo])
        except:
            continue
        
df2 = pd.DataFrame(result,columns=['NombreVueloOld','NombreVuelo'])
df2.head()

,NombreVueloOld,NombreVuelo
0,148_26_01_07_ED2,147_26_01_07_ED2
1,379_26_04_11_Camino_Alternativo_Salamanca,364_26_04_11_Camino_Alternativo_Salamanca
2,None,469_26_05_14_Camino_Alternativo_Salamanca
3,394_26_04_16_Camino_Alternativo_Salamanca,379_26_04_16_Camino_Alternativo_Salamanca
4,342_26_04_02_Camino_Alternativo_Salamanca,329_26_04_02_Camino_Alternativo_Salamanca


In [ ]:
query_update = tuple(df2.NombreVueloOld)
query_update = f'NombreVuelo IN {query_update}'
dict_rename = dict(zip(df2.NombreVueloOld,df2.NombreVuelo))

In [109]:
with arcpy.da.UpdateCursor(PATH_MOSAIC_DATASET,['NombreVuelo']) as cursor:
    for c in cursor:
        new_name_ = dict_rename.get(c[0],None)
        if new_name_:
            c[0] = new_name
            cursor.updateRow(c)
            

'147_26_01_07_ED2'

'148_26_01_07_ED2'

In [ ]:
aprx_path = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"
map_name = "CL MLP PAO 27 Imagenes Aereas PAO Image Server"
prefix = "CL_MLP_PAO_IF_Ortho_"
suffix = ".tif"
 
aprx = arcpy.mp.ArcGISProject(aprx_path)
maps = aprx.listMaps(map_name)
 
if not maps:
    print("No se encontró el mapa")
else:
    m = maps[0]
 
    # 1️⃣ Renombrar
    for lyr in m.listLayers():
        try:
            original_name = lyr.name
            new_name = original_name
            if new_name.startswith(prefix):
                new_name = new_name.replace(prefix, "", 1)
            if new_name.endswith(suffix):
                new_name = new_name[:-len(suffix)]
            if new_name != original_name:
                print(f"Renombrando: {original_name} -> {new_name}")
                lyr.name = new_name
        except Exception as e:
            print(f"Error con capa {lyr.name}: {e}")
 
#     # 2️⃣ Ordenar
#     layers = [lyr for lyr in m.listLayers() if lyr.isFeatureLayer or lyr.isRasterLayer]
#     layers_sorted = sorted(layers, key=lambda l: l.name)
#     for lyr in layers_sorted:
#         m.moveLayer(m.listLayers()[0], lyr, "BEFORE")
#     print("Capas ordenadas")
 
#     # 3️⃣ Guardar una sola vez
#     aprx.save()
#     print("Proyecto guardado")
 
# del aprx  # buena práctica siempre

In [19]:
maps

In [111]:
load_df.columns

Index(['ready_for_datastore', 'review_reason', 'path', 'relative_path',
       'file_name', 'expected_file_name', 'destination_path',
       'original_expected_file_name', 'expected_name', 'expected_date_token',
       'destination_folder', 'destination_date_folder', 'expected_sector',
       'sector_source', 'rename_status', 'spatial_status',
       'spatial_sector_raw', 'spatial_sector', 'spatial_overlap_pct',
       'spatial_overlap_count', 'spatial_all_matches',
       'duplicate_expected_file_name', 'duplicate_sequence',
       'duplicate_was_resolved', 'size_mb', 'modified_at', 'Name', 'Raster',
       'Path_Destino', 'Sector', 'Fecha_Adqui', 'URL', 'Proyecto', 'Sensor',
       'Fecha_Publ', 'relative_path_attr'],
      dtype='object')

In [114]:
load_df.loc[0,'destination_path']

'\\\\amssclgis10.ams.gmams.cl\\CL_MLP_PAO\\Chacay_Drone\\26_05\\CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera.tif'